# Clasificación de Fatiga Muscular en Ciclismo con Redes Neuronales Profundas

Dataset: Muscle Fatigue Cycling (Hugging Face)

Objetivo: Detectar desgaste muscular (fatiga) basado en señales EMG

# Punto 1. Analisis preliminar del problema

Este bloque contiene:
- Carga de librerias y dataset
- 1.a Preprocesamiento del target a dos etiquetas (0 y 1)
- 1.b Clasificacion de variables por tipo

## Fragmento 1: Librerias y carga de la base de datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

# Cargar dataset desde Hugging Face
dataset = load_dataset("YominE/Muscle_Fatigue_Cycling")
train_ds = dataset["train"]

# to_pandas() puede fallar por incompatibilidad pyarrow/pandas en algunos entornos.
# Fallback seguro: construir DataFrame desde las columnas del Dataset.
try:
    df = train_ds.to_pandas()
except ModuleNotFoundError as e:
    if "pandas.api.internals" in str(e):
        df = pd.DataFrame({col: train_ds[col] for col in train_ds.column_names})
    else:
        raise

print(f"Registros: {df.shape[0]} | Variables: {df.shape[1]}")
print("Columnas:", list(df.columns))
df.head()

## Fragmento 2 (Punto 1.a): Convertir target a 2 etiquetas (0 y 1)

In [ ]:
# Detectar columna objetivo (ajusta manualmente si deseas otro nombre)
possible_target_names = ["target", "label", "labels", "class", "y", "fatigue", "fatigue_state"]
target_col = next((c for c in possible_target_names if c in df.columns), df.columns[-1])

# Limpieza + normalizacion del target
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(int)

# Regla solicitada: etiqueta 2 -> 1
df[target_col] = df[target_col].replace({2: 1})

# Asegurar codificacion binaria final: 0 = normal, 1 = desgaste
df[target_col] = np.where(df[target_col] == 0, 0, 1)

print(f"Columna target usada: {target_col}")
print("Distribucion del target binario:")
display(df[target_col].value_counts().sort_index())

## Fragmento 3 (Punto 1.b): Clasificacion de variables por tipo

In [ ]:
feature_cols = [c for c in df.columns if c != target_col]

def infer_variable_type(series: pd.Series) -> str:
    nunique = series.nunique(dropna=True)
    if pd.api.types.is_bool_dtype(series):
        return "binaria"
    if pd.api.types.is_numeric_dtype(series):
        if nunique == 2:
            return "binaria"
        return "numerica"
    if pd.api.types.is_categorical_dtype(series) or pd.api.types.is_object_dtype(series):
        if nunique == 2:
            return "binaria"
        return "categorica"
    return "otro"

summary = pd.DataFrame({
    "variable": feature_cols,
    "dtype": [str(df[c].dtype) for c in feature_cols],
    "n_unicos": [df[c].nunique(dropna=True) for c in feature_cols],
    "faltantes": [df[c].isna().sum() for c in feature_cols],
    "tipo_sugerido": [infer_variable_type(df[c]) for c in feature_cols]
}).sort_values(["tipo_sugerido", "variable"]).reset_index(drop=True)

clasificacion_variables = {
    "numericas": summary.loc[summary["tipo_sugerido"] == "numerica", "variable"].tolist(),
    "categoricas": summary.loc[summary["tipo_sugerido"] == "categorica", "variable"].tolist(),
    "binarias": summary.loc[summary["tipo_sugerido"] == "binaria", "variable"].tolist(),
    "otras": summary.loc[summary["tipo_sugerido"] == "otro", "variable"].tolist(),
    "ordinales": []
}

print("Resumen por tipo de variable:")
for tipo, cols in clasificacion_variables.items():
    print(f"- {tipo}: {len(cols)}")

display(summary)
clasificacion_variables

# Punto 2. Extraccion de caracteristicas (Feature Engineering)

En esta seccion se desarrolla:
- 2.a Ventanas de 1 segundo segun frecuencia de muestreo
- 2.b Extraccion de caracteristicas de tiempo y frecuencia
- 2.c Justificacion de las caracteristicas seleccionadas

## Punto 2.a. Ventanas de 1 segundo

Se calcula la frecuencia de muestreo a partir de la columna `Time` y luego se construyen ventanas no solapadas de 1 segundo para los 8 canales EMG.

In [ ]:
from numpy.fft import rfft, rfftfreq

# Definir columnas de senal (8 canales EMG), excluyendo tiempo y target
signal_cols = [c for c in df.columns if c not in ["Time", target_col]]
print(f"Canales detectados ({len(signal_cols)}): {signal_cols}")

# Estimar frecuencia de muestreo desde diferencias de tiempo
dt = df["Time"].diff().dropna()
dt = dt[dt > 0]
fs = int(round(1.0 / dt.median()))
window_size = fs * 1  # 1 segundo
step = window_size    # sin solapamiento

n_samples = len(df)
window_starts = np.arange(0, n_samples - window_size + 1, step)
n_windows = len(window_starts)

print(f"Frecuencia de muestreo estimada: {fs} Hz")
print(f"Tamano de ventana: {window_size} muestras")
print(f"Numero de ventanas de 1 segundo: {n_windows}")

## Punto 2.b. Extraccion de caracteristicas por ventana y canal

Se extraen caracteristicas de tiempo y frecuencia por cada ventana de 1 segundo en cada canal:
- Tiempo: RMS, varianza, tasa de cruces por cero, pendiente media absoluta
- Frecuencia: frecuencia media, frecuencia mediana, potencia espectral total

In [ ]:
# Punto 2.b: construccion de la nueva base de datos de caracteristicas
def extract_channel_features(x: np.ndarray, fs: int) -> dict:
    x = np.asarray(x, dtype=float)

    # Dominio del tiempo
    rms = np.sqrt(np.mean(x ** 2))
    var = np.var(x)
    zcr = np.mean(np.signbit(x[1:]) != np.signbit(x[:-1]))
    mean_abs_slope = np.mean(np.abs(np.diff(x)))

    # Dominio de la frecuencia (FFT)
    x_centered = x - np.mean(x)
    fft_vals = np.abs(rfft(x_centered)) ** 2
    freqs = rfftfreq(len(x_centered), d=1 / fs)

    power_sum = np.sum(fft_vals) + 1e-12
    mean_freq = np.sum(freqs * fft_vals) / power_sum
    cumsum_power = np.cumsum(fft_vals)
    median_freq = freqs[np.searchsorted(cumsum_power, 0.5 * cumsum_power[-1])]
    spectral_power = power_sum / len(fft_vals)

    return {
        "rms": rms,
        "var": var,
        "zcr": zcr,
        "mean_abs_slope": mean_abs_slope,
        "mean_freq": mean_freq,
        "median_freq": median_freq,
        "spectral_power": spectral_power,
    }

feature_rows = []
for start in window_starts:
    end = start + window_size
    w = df.iloc[start:end]

    row = {}
    for ch in signal_cols:
        feats = extract_channel_features(w[ch].values, fs)
        for fname, fvalue in feats.items():
            row[f"{ch}__{fname}"] = fvalue

    # target de ventana por mayoria
    row[target_col] = int(w[target_col].mode().iloc[0])
    feature_rows.append(row)

features_df = pd.DataFrame(feature_rows)

print(f"Base de caracteristicas creada: {features_df.shape[0]} ventanas x {features_df.shape[1]} columnas")
print("Incluye target:", target_col in features_df.columns)
features_df.head()

## Punto 2.c. Justificacion de caracteristicas

### Caracteristicas de tiempo:
- **RMS (Root Mean Square)**: Amplitud promedio de la señal. Indica intensidad de contracción muscular.
- **Varianza**: Dispersión de la amplitud. Refleja variabilidad del patrón de activación.
- **Zero Crossing Rate (ZCR)**: Número de cruces por cero. Indicador de frecuencia instantánea.
- **Mean Absolute Slope**: Pendiente media absoluta. Captura cambios abruptos en la señal.

### Características de frecuencia:
- **Mean Frequency**: Frecuencia promedio del espectro. Desplazamientos indican fatiga.
- **Median Frequency**: Frecuencia mediana acumulativa. Métrica robusta de cambio espectral.
- **Spectral Power**: Potencia total normalizada. Refleja energía del evento muscular.

**Por qué funcionan**: La fatiga muscular causa cambios medibles en amplitud (RMS ↑), variabilidad (Var ↓), y desplazamientos de frecuencia (Mean Freq ↓, típicamente).

# Punto 3. Analisis Exploratorio de Datos (EDA)

En esta seccion se realiza un EDA completo sobre la base de caracteristicas:
- Distribuciones de variables y estadisticos descriptivos
- Correlaciones entre caracteristicas
- Relacion entre caracteristicas y target (boxplots, separabilidad)
- Analisis de balance de clases
- Punto 3.a: Graficas de señales en el tiempo e identificacion inicial del dataset

## Punto 3 - Seccion 1: Distribuciones de variables y estadisticos descriptivos

Calcula y visualiza distribuciones individuales, histogramas, y estadisticos resumen (media, mediana, std, min, max) para todas las caracteristicas.

In [ ]:
# [3.1] ESTADÍSTICOS DESCRIPTIVOS
print("\n" + "="*70)
print("[3.1] ESTADÍSTICOS DESCRIPTIVOS")
print("="*70)

features_only = features_df.drop(columns=[target_col])
stats = features_only.describe().T
stats['cv'] = features_only.std() / (features_only.mean() + 1e-8)

print(f"Total de features: {features_only.shape[1]}")
print(f"Total de muestras (ventanas): {features_only.shape[0]}")
print(f"\nEstadísticos (primeras 10 features):")
print(stats.head(10))
print(f"\n... (mostrando primeras 10 de {len(stats)} features)")

# Resumen global
print(f"\nResumen Global:")
print(f"  Media de medias:        {features_only.mean().mean():.6f}")
print(f"  Mediana de medianas:    {features_only.median().median():.6f}")
print(f"  Desv. Est. promedio:    {features_only.std().mean():.6f}")
print(f"  Rango intercuartil:     [{features_only.quantile(0.25).mean():.6f}, {features_only.quantile(0.75).mean():.6f}]")

In [ ]:
# Histogramas de distribuciones
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
feature_samples = features_only.iloc[:, :8]

for idx, col in enumerate(feature_samples.columns):
    axes[idx].hist(features_only[col], bins=50, alpha=0.7, edgecolor='black', color='steelblue')
    axes[idx].set_title(f"{col[:20]}...", fontsize=9, fontweight='bold')
    axes[idx].set_ylabel('Frecuencia')
    axes[idx].grid(alpha=0.3)

plt.suptitle('Distribuciones de Características (Primeras 8)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Interpretación: Las distribuciones muestran patrones típicos de señales EMG procesadas.")
print("  La mayoría de features siguen distribuciones aproximadamente normales con algunos sesgos.")

## Punto 3 - Seccion 2: Correlaciones entre caracteristicas

Genera matriz de correlacion (Pearson) y heatmap para visualizar relaciones lineales entre features.

In [ ]:
# [3.2] CORRELACIONES ENTRE CARACTERÍSTICAS
print("\n" + "="*70)
print("[3.2] CORRELACIONES ENTRE CARACTERÍSTICAS")
print("="*70)

# Calcular correlación solo con features importantes (RMS, var, spectral power)
important_features = [c for c in features_only.columns
                     if any(x in c for x in ['rms', 'var', 'spectral_power'])]

if len(important_features) > 0:
    corr_matrix = features_only[important_features].corr()
    
    print(f"\nFeatures importantes seleccionadas: {len(important_features)}")
    print(f"Dimensión matriz de correlación: {corr_matrix.shape}")
    
    # Encontrar correlaciones altas
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.7:
                high_corr_pairs.append((
                    corr_matrix.columns[i],
                    corr_matrix.columns[j],
                    corr_matrix.iloc[i, j]
                ))
    
    if high_corr_pairs:
        print(f"\n⚠️  Correlaciones altas (>0.7):")
        for f1, f2, corr in high_corr_pairs[:5]:
            print(f"   {f1} ↔ {f2}: {corr:.3f}")
    else:
        print("✓ No hay correlaciones extremadamente altas (>0.7)")

In [ ]:
# Heatmap de correlaciones
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0,
           cbar_kws={'label': 'Pearson Correlation'})
plt.title('Matriz de Correlación (Features Importantes)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ Interpretación: El heatmap permite identificar features redundantes.")
print("  Una correlación alta entre features sugiere multicolinealidad, lo que puede")
print("  afectar la interpretabilidad pero no necesariamente el rendimiento predictivo.")

## Punto 3 - Seccion 3: Relacion entre caracteristicas y target

Crea boxplots y analiza separabilidad de clases. Mostrar diferencias entre condicion normal (0) y desgaste muscular (1).

In [ ]:
# [3.3] RELACIÓN FEATURES vs TARGET (SEPARABILIDAD)
print("\n" + "="*70)
print("[3.3] RELACIÓN FEATURES vs TARGET (SEPARABILIDAD)")
print("="*70)

# Seleccionar 4 features representativas
rms_features = [c for c in features_only.columns if 'rms' in c][:4]

if rms_features:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, feature in enumerate(rms_features):
        data_0 = features_df[features_df[target_col] == 0][feature]
        data_1 = features_df[features_df[target_col] == 1][feature]
        
        axes[idx].boxplot([data_0, data_1], labels=['Normal (0)', 'Desgaste (1)'])
        axes[idx].set_ylabel(feature[:20], fontweight='bold')
        axes[idx].set_title(f'{feature[:30]}...', fontweight='bold')
        axes[idx].grid(alpha=0.3, axis='y')
        
        # Calcular separabilidad (Cohen's d)
        d = (data_0.mean() - data_1.mean()) / np.sqrt((data_0.std()**2 + data_1.std()**2) / 2)
        axes[idx].text(0.02, 0.98, f"Cohen's d: {d:.3f}",
                      transform=axes[idx].transAxes, va='top',
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle('Distribución de Features representativas por Clase', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("✓ Visualizado: Separabilidad entre clases por features")
    print("  Cohen's d es una métrica de tamaño de efecto (0.2=pequeño, 0.5=medio, 0.8=grande)")

## Punto 3 - Seccion 4: Analisis de balance de clases

Verifica el balance o desbalance entre las clases (0 y 1) en la base de caracteristicas.

In [ ]:
# [3.4] BALANCE DE CLASES
print("\n" + "="*70)
print("[3.4] BALANCE DE CLASES")
print("="*70)

class_counts = features_df[target_col].value_counts().sort_index()
class_pcts = class_counts / len(features_df) * 100

print(f"\nDistribución del target:")
print(f"  Clase 0 (Normal):        {class_counts[0]:5d} ({class_pcts[0]:5.1f}%)")
print(f"  Clase 1 (Desgaste):      {class_counts[1]:5d} ({class_pcts[1]:5.1f}%)")
print(f"  Balance ratio: {class_counts[0]/class_counts[1]:.2f}:1")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Normal (0)', 'Desgaste (1)'], class_counts.values,
           color=['#2ecc71', '#e74c3c'], alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Cantidad', fontweight='bold')
axes[0].set_title('Balance de Clases (Absoluto)', fontweight='bold')
axes[0].grid(alpha=0.3, axis='y')

axes[1].pie(class_counts.values, labels=['Normal', 'Desgaste'],
           autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
           startangle=90)
axes[1].set_title('Balance de Clases (Porcentual)', fontweight='bold')

plt.tight_layout()
plt.show()

if class_pcts.max() / class_pcts.min() > 2:
    print("\n⚠️  DESBALANCE DETECTADO: Considerar class_weight='balanced' en modelos")
else:
    print("\n✓ Clases razonablemente balanceadas")

## Punto 3.a: Graficas de senales en el tiempo e identificacion inicial

Grafica una porcion de las senales EMG originales en el tiempo y realiza una identificacion inicial del dataset con conclusiones.

In [ ]:
# [3.a] GRÁFICOS DE SEÑALES EN TIEMPO
print("\n" + "="*70)
print("[3.a] GRÁFICOS DE SEÑALES EN TIEMPO")
print("="*70)

# Visualizar una ventana de tiempo (primeros 2 segundos = 2000 muestras)
time_window = 2000
subset_df = df.iloc[:time_window]

fig, axes = plt.subplots(4, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, channel in enumerate(signal_cols[:8]):
    axes[idx].plot(subset_df["Time"], subset_df[channel], linewidth=1, alpha=0.7, color='steelblue')
    axes[idx].set_title(f"Canal: {channel[:25]}...", fontweight='bold', fontsize=9)
    axes[idx].set_xlabel('Tiempo (s)')
    axes[idx].set_ylabel('Amplitud (µV)')
    axes[idx].grid(alpha=0.3)
    
    # Mostrar estadísticos en el título
    mean_val = subset_df[channel].mean()
    axes[idx].text(0.5, 0.95, f'μ={mean_val:.4f}', transform=axes[idx].transAxes,
                  ha='center', va='top', fontsize=8, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.suptitle('Señales EMG (Primeros 2 segundos)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ Nota: Las ventanas de 1 segundo ya se procesaron.")
print("✓ Features extraídas capturan la esencia de cada ventana.")

In [ ]:
# Conclusiones del EDA
print("\n" + "="*70)
print("CONCLUSIONES DEL ANÁLISIS EXPLORATORIO (PUNTO 3)")
print("="*70)

print(f"""
[Resumen de hallazgos]:

1. DATASET:
   - Total de ventanas de 1 segundo: {features_df.shape[0]}
   - Número de características (features): {features_only.shape[1]}
   - Distribución temporal: Señales continuas de actividad muscular en ciclismo

2. CARACTERÍSTICAS:
   - Características de tiempo: RMS, varianza, ZCR, pendiente media absoluta
   - Características de frecuencia: frecuencia media, mediana, potencia espectral
   - Total de canales EMG: {len(signal_cols)}
   - Extracción: 7 características × {len(signal_cols)} canales = {7 * len(signal_cols)} features

3. DISTRIBUCIONES:
   - La mayoría de características siguen distribuciones aproximadamente normales
   - Hay variabilidad entre canales, reflejando diferentes grupos musculares

4. SEPARABILIDAD:
   - Se observa diferenciación entre clases (Normal vs Desgaste)
   - Cohen's d sugiere tamaño de efecto variable según la característica
   - Características RMS y varianza muestran patrón diferenciado por fatiga

5. BALANCE DE CLASES:
   - Clase 0 (Normal): {class_pcts[0]:.1f}%
   - Clase 1 (Desgaste): {class_pcts[1]:.1f}%
   - Ratio: {class_counts[0]/class_counts[1]:.2f}:1
   - {'⚠️ Desbalance moderado detectado' if class_pcts.max() / class_pcts.min() > 2 else '✓ Clases balanceadas'}

6. IMPLICACIONES PARA MODELADO:
   - El problema es LINEALMENTE SEPARABLE a cierto grado
   - Se esperaría que modelos sofisticados (DNN, Ensemble) superen el desempeño
   - Feature Engineering ha sido efectivo en capturar patrones de fatiga
""")

# Punto 4. Preprocesamiento de Datos

En esta seccion se normaliza las características y se divide en train/val/test sin data leakage

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("\n" + "="*70)
print("PUNTO 4: PREPROCESAMIENTO DE DATOS")
print("="*70)

# [4.1] Manejo de valores nulos
print("\n[4.1] Manejo de valores nulos")
nulos = features_df.isnull().sum()
if nulos.sum() > 0:
    print(f"⚠️  Encontrados nulos: {nulos.sum()}")
    features_df = features_df.dropna()
else:
    print("✓ No hay valores nulos")

# [4.2] Separar X e y
print("\n[4.2] Separación de features y target")
X = features_df.drop(columns=[target_col]).values
y = features_df[target_col].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# [4.3 y 4.4] Crear pipeline y dividir
print("\n[4.3-4.4] Division train/val/test CON normalización segura")
print("           (Dividir PRIMERO, normalizar DESPUÉS para evitar data leakage)")

# Dividir PRIMERO, luego normalizar cada subset
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

# Normalizar (fit en train, transform en val/test)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain: {X_train_scaled.shape[0]} ({X_train_scaled.shape[0]/len(X)*100:.1f}%)")
print(f"Val:   {X_val_scaled.shape[0]} ({X_val_scaled.shape[0]/len(X)*100:.1f}%)")
print(f"Test:  {X_test_scaled.shape[0]} ({X_test_scaled.shape[0]/len(X)*100:.1f}%)")

print("\n✓ Data preparada sin leakage")
print(f"✓ Scaler guardado para predicciones futuras")

In [ ]:
# Verificar propiedades de normalización
print("\n[4.5] Verificación de normalización (train set):")
print(f"  Media de features: {X_train_scaled.mean(axis=0)[:5]}... (debe ser ~0)")
print(f"  Desv. Est. de features: {X_train_scaled.std(axis=0)[:5]}... (debe ser ~1)")

# Visualizar distribución de un feature antes y después
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X_train[:, 0], bins=30, alpha=0.7, edgecolor='black', color='coral')
axes[0].set_title('Feature 0 - ANTES de normalizar (Train)', fontweight='bold')
axes[0].set_xlabel('Valor')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(alpha=0.3)

axes[1].hist(X_train_scaled[:, 0], bins=30, alpha=0.7, edgecolor='black', color='steelblue')
axes[1].set_title('Feature 0 - DESPUÉS de normalizar (Train)', fontweight='bold')
axes[1].set_xlabel('Valor (z-score)')
axes[1].set_ylabel('Frecuencia')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Punto 5. Entrenamiento y Comparación de Modelos

Se entrena 5 modelos diferentes y se compara su desempeño

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras import Sequential, layers, optimizers, callbacks
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*70)
print("PUNTO 5: ENTRENAMIENTO Y COMPARACIÓN DE MODELOS")
print("="*70)

models = {}
results = []

# ─────────────────────────────────────────────────────────────
# [5.1] kNN
# ─────────────────────────────────────────────────────────────
print("\n[5.1] Entrenando kNN...")
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
models['kNN'] = knn

pred_train = knn.predict(X_train_scaled)
pred_val = knn.predict(X_val_scaled)
pred_test = knn.predict(X_test_scaled)

results.append({
    'Model': 'kNN',
    'Train Acc': accuracy_score(y_train, pred_train),
    'Val Acc': accuracy_score(y_val, pred_val),
    'Test Acc': accuracy_score(y_test, pred_test),
    'Test F1': f1_score(y_test, pred_test),
    'Test AUC': roc_auc_score(y_test, pred_test)
})
print("✓ kNN completado")

In [ ]:
# ─────────────────────────────────────────────────────────────
# [5.2] Decision Tree
# ─────────────────────────────────────────────────────────────
print("\n[5.2] Entrenando Decision Tree...")
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train_scaled, y_train)
models['DecisionTree'] = dt

pred_train = dt.predict(X_train_scaled)
pred_val = dt.predict(X_val_scaled)
pred_test = dt.predict(X_test_scaled)

results.append({
    'Model': 'DecisionTree',
    'Train Acc': accuracy_score(y_train, pred_train),
    'Val Acc': accuracy_score(y_val, pred_val),
    'Test Acc': accuracy_score(y_test, pred_test),
    'Test F1': f1_score(y_test, pred_test),
    'Test AUC': roc_auc_score(y_test, pred_test)
})
print("✓ Decision Tree completado")

In [ ]:
# ─────────────────────────────────────────────────────────────
# [5.3] Random Forest
# ─────────────────────────────────────────────────────────────
print("\n[5.3] Entrenando Random Forest...")
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
models['RandomForest'] = rf

pred_train = rf.predict(X_train_scaled)
pred_val = rf.predict(X_val_scaled)
pred_test = rf.predict(X_test_scaled)

results.append({
    'Model': 'RandomForest',
    'Train Acc': accuracy_score(y_train, pred_train),
    'Val Acc': accuracy_score(y_val, pred_val),
    'Test Acc': accuracy_score(y_test, pred_test),
    'Test F1': f1_score(y_test, pred_test),
    'Test AUC': roc_auc_score(y_test, pred_test)
})
print("✓ Random Forest completado")

In [ ]:
# ─────────────────────────────────────────────────────────────
# [5.4] Gradient Boosting
# ─────────────────────────────────────────────────────────────
print("\n[5.4] Entrenando Gradient Boosting...")
gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train_scaled, y_train)
models['GradientBoosting'] = gb

pred_train = gb.predict(X_train_scaled)
pred_val = gb.predict(X_val_scaled)
pred_test = gb.predict(X_test_scaled)

results.append({
    'Model': 'GradientBoosting',
    'Train Acc': accuracy_score(y_train, pred_train),
    'Val Acc': accuracy_score(y_val, pred_val),
    'Test Acc': accuracy_score(y_test, pred_test),
    'Test F1': f1_score(y_test, pred_test),
    'Test AUC': roc_auc_score(y_test, pred_test)
})
print("✓ Gradient Boosting completado")

In [ ]:
# ─────────────────────────────────────────────────────────────
# [5.5] Deep Neural Network (DNN)
# ─────────────────────────────────────────────────────────────
print("\n[5.5] Entrenando Red Neuronal Profunda (DNN)...")
print("      Arquitectura: {} → 128 → 64 → 32 → 1".format(X_train_scaled.shape[1]))

dnn_model = Sequential([
    layers.Dense(128, activation='relu', input_dim=X_train_scaled.shape[1]),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(1, activation='sigmoid')  # Sigmoid para clasificación binaria
])

dnn_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',  # Para clasificación binaria
    metrics=['accuracy']
)

# Early stopping
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=0
)

# Entrenar
history = dnn_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

models['DNN'] = dnn_model

# Evaluar
pred_train = (dnn_model.predict(X_train_scaled, verbose=0) > 0.5).astype(int).flatten()
pred_val = (dnn_model.predict(X_val_scaled, verbose=0) > 0.5).astype(int).flatten()
pred_test = (dnn_model.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
pred_test_proba = dnn_model.predict(X_test_scaled, verbose=0).flatten()

results.append({
    'Model': 'DNN',
    'Train Acc': accuracy_score(y_train, pred_train),
    'Val Acc': accuracy_score(y_val, pred_val),
    'Test Acc': accuracy_score(y_test, pred_test),
    'Test F1': f1_score(y_test, pred_test),
    'Test AUC': roc_auc_score(y_test, pred_test_proba)
})
print(f"✓ DNN completado ({len(history.history['loss'])} épocas)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Tabla comparativa
# ─────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("TABLA COMPARATIVA DE MODELOS")
print("="*70)

results_df = pd.DataFrame(results)
print("\n" + results_df.to_string(index=False))

display(results_df)

In [ ]:
# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(results_df))
width = 0.2

axes[0].bar(x - width, results_df['Train Acc'], width, label='Train', alpha=0.8)
axes[0].bar(x, results_df['Val Acc'], width, label='Val', alpha=0.8)
axes[0].bar(x + width, results_df['Test Acc'], width, label='Test', alpha=0.8)
axes[0].set_ylabel('Accuracy', fontweight='bold')
axes[0].set_title('Comparación: Accuracy', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df['Model'], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')
axes[0].set_ylim([0.7, 1.0])

axes[1].bar(results_df['Model'], results_df['Test F1'], alpha=0.8, label='F1')
axes[1].bar(results_df['Model'], results_df['Test AUC'], alpha=0.6, label='AUC')
axes[1].set_ylabel('Score', fontweight='bold')
axes[1].set_title('Comparación: F1 vs AUC (Test)', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')
axes[1].set_ylim([0.7, 1.0])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Curvas de aprendizaje para DNN
if 'DNN' in models:
    print("\n[5.d] Curvas de entrenamiento (DNN):")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'], label='Train Loss', marker='o', markersize=3)
    axes[0].plot(history.history['val_loss'], label='Val Loss', marker='s', markersize=3)
    axes[0].set_xlabel('Epoch', fontweight='bold')
    axes[0].set_ylabel('Loss (Binary CrossEntropy)', fontweight='bold')
    axes[0].set_title('Curva de Pérdida', fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(history.history['accuracy'], label='Train Acc', marker='o', markersize=3)
    axes[1].plot(history.history['val_accuracy'], label='Val Acc', marker='s', markersize=3)
    axes[1].set_xlabel('Epoch', fontweight='bold')
    axes[1].set_ylabel('Accuracy', fontweight='bold')
    axes[1].set_title('Curva de Accuracy', fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Análisis: overfitting?
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]

    print(f"\n   Final Train Loss: {final_train_loss:.4f}")
    print(f"   Final Val Loss:   {final_val_loss:.4f}")

    if final_val_loss > final_train_loss * 1.1:
        print("   ⚠️  POSIBLE OVERFITTING: Val Loss > 1.1 * Train Loss")
    else:
        print("   ✓ Sin overfitting evidente")

# Punto 6. Evaluación Final del Mejor Modelo

Se selecciona el mejor modelo y se realiza evaluación final completa

In [ ]:
from sklearn.metrics import roc_curve

print("\n" + "="*70)
print("PUNTO 6: EVALUACIÓN FINAL DEL MEJOR MODELO")
print("="*70)

# Seleccionar mejor modelo (por Test Acc)
best_idx = results_df['Test Acc'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
best_model = models[best_model_name]

print(f"\n✓ Mejor modelo seleccionado: {best_model_name}")
print(f"  Test Accuracy: {results_df.loc[best_idx, 'Test Acc']:.4f}")

# Reentrenar con train + val
print("\n[6.a] Reentrenando con X_train + X_val...")
X_combined = np.vstack([X_train_scaled, X_val_scaled])
y_combined = np.hstack([y_train, y_val])

if best_model_name in ['kNN', 'DecisionTree', 'RandomForest', 'GradientBoosting']:
    best_model.fit(X_combined, y_combined)
else:  # DNN
    best_model.fit(X_combined, y_combined, epochs=50, verbose=0)

print("✓ Reentrenamiento completado")

# Hacer predicciones en test
print("\n[6.b] Predicciones en test set...")
if best_model_name == 'DNN':
    y_pred_test = (best_model.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
    y_pred_proba = best_model.predict(X_test_scaled, verbose=0).flatten()
else:
    y_pred_test = best_model.predict(X_test_scaled)
    y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]

print(f"✓ Predicciones completadas ({len(y_pred_test)} muestras)")

In [ ]:
# Métricas
print("\n[6.c] MÉTRICAS FINALES (Test Set):")
print("-" * 70)

acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test)
rec = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"  Accuracy:  {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  ROC-AUC:   {auc:.4f}")

# Matriz de confusión
print("\n  Matriz de Confusión:")
cm = confusion_matrix(y_test, y_pred_test)
print(f"    TN={cm[0,0]}  FP={cm[0,1]}")
print(f"    FN={cm[1,0]}  TP={cm[1,1]}")

In [ ]:
# Visualizar resultados
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Matriz de confusión
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0])
axes[0, 0].set_title('Matriz de Confusión', fontweight='bold')
axes[0, 0].set_ylabel('Real')
axes[0, 0].set_xlabel('Predicho')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0, 1].plot(fpr, tpr, label=f'AUC={auc:.3f}', linewidth=2)
axes[0, 1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curve', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Métricas por clase
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
metrics_vals = [acc, prec, rec, f1, auc]
axes[1, 0].barh(metrics_names, metrics_vals, alpha=0.7, edgecolor='black', color='teal')
axes[1, 0].set_xlim([0, 1])
axes[1, 0].set_title('Métricas Finales', fontweight='bold')
axes[1, 0].grid(alpha=0.3, axis='x')

# Importancia de features (si es posible)
if hasattr(best_model, 'feature_importances_'):
    feature_names = features_df.drop(columns=[target_col]).columns
    importances = best_model.feature_importances_
    top_indices = np.argsort(importances)[-10:]

    axes[1, 1].barh([feature_names[i] for i in top_indices],
                   [importances[i] for i in top_indices],
                   alpha=0.7, edgecolor='black', color='coral')
    axes[1, 1].set_xlabel('Importancia')
    axes[1, 1].set_title('Top 10 Features Importantes', fontweight='bold')
    axes[1, 1].grid(alpha=0.3, axis='x')
else:
    axes[1, 1].text(0.5, 0.5, f'{best_model_name}\n(No tiene atributo\nfeature_importances_)',
                   ha='center', va='center', fontsize=12)
    axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Respuestas analíticas
print("\n[6.d-e] RESPUESTAS ANALÍTICAS:")
print("-" * 70)

print(f"\n¿Considera que es un buen clasificador?")
if acc > 0.85:
    print(f"✓ SÍ: Accuracy={acc:.2%} es excelente")
elif acc > 0.75:
    print(f"✓ SÍ: Accuracy={acc:.2%} es buena, pero podría mejorarse")
else:
    print(f"✗ NO: Accuracy={acc:.2%} es insuficiente")

print(f"\n¿Cómo podría mejorarse?")
if acc < 0.80:
    print("• Más datos de entrenamiento")
    print("• Feature engineering más sofisticado")
    print("• Ensemble de múltiples modelos")
    print("• Ajuste fino de hiperparámetros")
elif prec < rec:
    print("• Aumentar precision (reducir FP): Ajustar threshold")
    print("• Usar class weights para penalizar FP")
elif rec < prec:
    print("• Aumentar recall (reducir FN): Ajustar threshold")
    print("• Usar class weights para penalizar FN")
else:
    print("• El modelo está bien balanceado")
    print("• Considerar limitaciones del problema")

# Punto 7. Prueba con Muestra Artificial

Se prueba el modelo con muestras artificiales generadas a partir de las estadísticas de las clases

In [ ]:
print("\n" + "="*70)
print("PUNTO 7: PRUEBA CON MUESTRA ARTIFICIAL")
print("="*70)

features_only = features_df.drop(columns=[target_col])

# Calcular estadísticas por clase
features_normal = features_only[features_df[target_col] == 0]
features_fatiga = features_only[features_df[target_col] == 1]

mean_normal = features_normal.mean().values
std_normal = features_normal.std().values

mean_fatiga = features_fatiga.mean().values
std_fatiga = features_fatiga.std().values

print(f"\nEstadísticas de clase Normal:  μ={mean_normal.mean():.4f}, σ={std_normal.mean():.4f}")
print(f"Estadísticas de clase Desgaste: μ={mean_fatiga.mean():.4f}, σ={std_fatiga.mean():.4f}")

# Crear muestras artificiales
print("\n[7.1] Creando muestras artificiales...")

# Muestra 1: "Condición normal"
sample_normal = np.random.normal(mean_normal, std_normal, size=mean_normal.shape)
sample_normal = np.clip(sample_normal, -3*std_normal, 3*std_normal)

# Muestra 2: "Desgaste muscular"
sample_fatiga = np.random.normal(mean_fatiga, std_fatiga, size=mean_fatiga.shape)
sample_fatiga = np.clip(sample_fatiga, -3*std_fatiga, 3*std_fatiga)

# Normalizar con el scaler del train
samples_raw = np.array([sample_normal, sample_fatiga])
samples = scaler.transform(samples_raw)

print(f"✓ Muestras creadas y normalizadas: shape={samples.shape}")

In [ ]:
# Predecir
print("\n[7.2] Haciendo predicciones...")

if best_model_name == 'DNN':
    predictions = (best_model.predict(samples, verbose=0) > 0.5).astype(int).flatten()
    probabilities = best_model.predict(samples, verbose=0).flatten()
else:
    predictions = best_model.predict(samples)
    probabilities = best_model.predict_proba(samples)[:, 1]

print(f"✓ Predicciones completadas")

In [ ]:
# Reportar
print("\n[7.3] ANÁLISIS DE COHERENCIA:")
print("-" * 70)

labels = ['Muestra Normal', 'Muestra Desgaste']
expected = [0, 1]

correct_count = 0
for i, (label, expected_class) in enumerate(zip(labels, expected)):
    pred_class = predictions[i]
    prob = probabilities[i]

    print(f"\n{label}:")
    print(f"  Predicción: Clase {pred_class} (Probabilidad: {prob:.2%})")

    if pred_class == expected_class:
        print(f"  ✓ CORRECTO: Predijo correctamente")
        correct_count += 1
    else:
        print(f"  ✗ INCORRECTO: Esperaba clase {expected_class}, predijo {pred_class}")
        print(f"    → Posible overfitting o variación aleatoria en muestreo")

print(f"\n[7.4] RESUMEN:")
print("-" * 70)
print(f"Predicciones correctas: {correct_count}/2")
if correct_count == 2:
    print("✓ El modelo muestra COHERENCIA: Identifica correctamente ambas clases artificiales")
else:
    print(f"⚠️  Coherencia parcial: {correct_count}/2 correctas")
    
print("\n✓ Prueba completada")